# Berlin parcel import — data-quality audit

## tl;dr

This notebook reruns the citywide source-extract audit and presents the acceptance checks for the parcel dashboard. The detailed values below are generated from the current local extracts, not copied manually.

## Context & Methods

**Intended grain:** one official ALKIS Flurstück per parcel row, with zero or more exact B-Plan scope intersections. Required parcel keys and geometry must be complete and unique. Official area values of zero are retained for provenance but are not safe for capacity calculations.

### Key Assumptions

- The WFS `numberMatched` value is the expected extraction volume.
- B-Plan scope overlap is evidence, not proof that every overlapping plan controls.
- GRZ, GFZ, storeys and permitted uses are outside this ingestion audit because the scope WFS does not publish those rules.

## Data

Rerun the inspectable streaming audit against the citywide NDJSON extracts.

In [1]:
import json, subprocess
from pathlib import Path

project = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
output = project / 'data/qa/citywide-import-audit.json'
subprocess.run(['node', 'scripts/audit-extracts.mjs', f'--output={output}'], cwd=project, check=True, capture_output=True, text=True)
audit = json.loads(output.read_text())
{'generatedAt': audit['generatedAt'], 'grain': audit['grain'], 'sources': audit['sources']}

{'generatedAt': '2026-08-01T08:47:36.905Z',
 'grain': 'one ALKIS Flurstück per parcel row; zero or more B-Plan scope segments per parcel',
 'sources': {'parcelsPath': 'data/import/alkis-parcels.ndjson',
  'segmentsPath': 'data/import/parcel-planning-segments.ndjson'}}

## Results

The compact acceptance profile and finding list are shown below.

In [2]:
parcel = audit['parcels']
segments = audit['planningSegments']
{
  'parcel_rows': parcel['rows'],
  'distinct_parcel_ids': parcel['distinctIds'],
  'duplicate_ids': parcel['duplicateIds'],
  'invalid_geometry_or_centroid': parcel['invalidGeometry'] + parcel['invalidCentroid'],
  'non_positive_official_area': parcel['nonPositiveArea'],
  'planning_intersections': segments['rows'],
  'covered_parcels': segments['coveredParcels'],
  'uncovered_parcels': segments['uncoveredParcels'],
  'covered_rate_pct': round(segments['coveredParcelRate'] * 100, 2),
  'distinct_plan_scopes': segments['distinctPlans'],
  'bnp_raster_candidates': audit['baunutzungsplanCandidates']['classified'],
  'bnp_samples_withheld': audit['baunutzungsplanCandidates']['withheld'],
  'fluchtlinie_relations': audit['fluchtlinienRelations']['rows'],
  'parcels_with_fluchtlinie_evidence': audit['fluchtlinienRelations']['parcels'],
}

{'parcel_rows': 403484,
 'distinct_parcel_ids': 403484,
 'duplicate_ids': 0,
 'invalid_geometry_or_centroid': 0,
 'non_positive_official_area': 10,
 'planning_intersections': 151672,
 'covered_parcels': 119942,
 'uncovered_parcels': 283542,
 'covered_rate_pct': 29.73,
 'distinct_plan_scopes': 2840,
 'bnp_raster_candidates': 127302,
 'bnp_samples_withheld': 77211,
 'fluchtlinie_relations': 168310,
 'parcels_with_fluchtlinie_evidence': 76930}

In [3]:
audit['findings']

[{'severity': 'pass', 'check': 'primary-key uniqueness', 'affectedRows': 0},
 {'severity': 'pass', 'check': 'geometry validity', 'affectedRows': 0},
 {'severity': 'medium',
  'check': 'positive official area',
  'affectedRows': 10,
  'action': 'Keep official value but exclude from capacity calculations pending source review.'},
 {'severity': 'pass',
  'check': 'spatial-intersection validity',
  'affectedRows': 0},
 {'severity': 'medium',
  'check': 'Baunutzungsplan raster classification',
  'affectedRows': 127302,
  'action': 'Treat as candidates only; resolve ambiguous classes, Baustufe boundaries, BO 1958 and Fluchtlinien before filling legal profile values.'}]

## Takeaways

- Primary keys, UUIDs, boroughs, geometry and centroids must pass with zero failures before loading the dashboard database.
- Zero-area source records remain visible but must be excluded from footprint and floor-area calculations.
- Parcels without an imported in-force B-Plan scope are not automatically buildable or governed by §34; they require the separate Baunutzungsplan, building-line and §§34/35 workflow.
- Multiple intersections per parcel are expected and must be resolved using plan status, effective dates and supersession relations.